# Data Retrieval Pipeline: VIX, VVIX, SVXY, and SVIX
**MScFE 690 Capstone -Group 17843**

**Authors:** Edgar NAVA-edgar.nava@grovewm.com & Celestin NYANDWI-celestinnyandwi48@gmail.com

**Purpose:** This notebook downloads historical daily OHLC data from Yahoo Finance for the VIX, VVIX, SVXY, and SVIX, cleans the datasets, and exports them as CSVs to the configured `data/raw/` folder for the Capstone analyses. The study end date is fixed at September 16, 2026.

### 1. Environment Setup & Dependencies
Ensure all required external libraries are installed for the local environment.

In [1]:
# Install necessary packages (run once)
!pip install -q pandas yfinance

### 2. Global Configuration & Local Directory Setup
Use the `data/raw/` directory defined in the project `config.json` for saving the final CSVs. Define the global study parameters, including the fixed end date of September 16, 2026, and the required OHLC columns to standardize the outputs.

In [1]:
from pathlib import Path
import json
import pandas as pd
import yfinance as yf

# Read the shared data paths from the project configuration.
PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "config.json").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Run this notebook from within the project folder.")
CONFIG = json.loads((PROJECT_ROOT / "config.json").read_text(encoding="utf-8"))
RAW_DATA_DIR = PROJECT_ROOT / CONFIG["raw_data_dir"]
PROCESSED_DATA_DIR = PROJECT_ROOT / CONFIG["processed_data_dir"]

OUTPUT_FOLDER = RAW_DATA_DIR
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

# Fixed study end date
END_DATE = pd.Timestamp("2026-09-16")

# Required columns for the final output
OUTPUT_COLUMNS = ["Date", "Open", "High", "Low", "Close"]

print(f"Environment ready. Data will be saved locally")

Environment ready. Data will be saved locally


### 3. VIX and VVIX Index Retrieval Engine
Define the function to retrieve historical daily OHLC data from the Yahoo Finance API. The engine strips timezone data, drops rows with missing values, rounds price points to six decimal places for precision, and exports the cleaned datasets to the local directory.

**VVIX data note:** The Yahoo Finance VVIX dataset had eight missing observations. We added only their Close values manually from Cboe's historical data, without interpolation. Open, High, and Low remain blank for those dates. This manual addition is not included in the download code below. The download is saved as `VVIX_incomplete.csv` to preserve the manually completed `VVIX_data.csv` used for analysis.

In [2]:
# Configuration for VIX and VVIX
INDEX_CONFIG = {
    "VVIX": {
        "ticker": "^VVIX",
        "start_date": pd.Timestamp("2007-01-03"),
        "output_file": "VVIX_incomplete.csv",
    },
    "VIX": {
        "ticker": "^VIX",
        "start_date": pd.Timestamp("2004-03-26"),
        "output_file": "VIX_data.csv",
    },
}

def download_index_data(index_name, ticker_symbol, start_date, end_date, output_file):
    print(f"Downloading {index_name} data...")
    yahoo_end_date = end_date + pd.Timedelta(days=1)
    ticker = yf.Ticker(ticker_symbol)

    data = ticker.history(
        start=start_date.strftime("%Y-%m-%d"),
        end=yahoo_end_date.strftime("%Y-%m-%d"),
        interval="1d",
        auto_adjust=False,
        actions=False,
    )

    if data.empty:
        raise ValueError(f"No historical data returned for {index_name}.")

    if data.index.tz is not None:
        data.index = data.index.tz_localize(None)

    data = data.reset_index()
    data = data.rename(columns={data.columns[0]: "Date"})
    data["Date"] = pd.to_datetime(data["Date"], errors="coerce")

    data = data.loc[(data["Date"] >= start_date) & (data["Date"] <= end_date)].copy()
    data = data.dropna(subset=OUTPUT_COLUMNS)
    data = data.sort_values("Date").reset_index(drop=True)

    data = data[OUTPUT_COLUMNS]
    price_columns = ["Open", "High", "Low", "Close"]
    data[price_columns] = data[price_columns].round(6)
    data["Date"] = data["Date"].dt.strftime("%Y-%m-%d")

    output_path = OUTPUT_FOLDER / output_file
    data.to_csv(output_path, index=False, encoding="utf-8")

    print(f"Saved {index_name} | Observations: {len(data):,}")
    return data

# Execute downloads
historical_data = {}
for index_name, config in INDEX_CONFIG.items():
    historical_data[index_name] = download_index_data(
        index_name, config["ticker"], config["start_date"], END_DATE, config["output_file"]
    )

Saved VVIX | Observations: 4,948
Saved VIX | Observations: 5,656


### 4. SVXY and SVIX ETF Retrieval
Download both ETFs from Yahoo Finance through the fixed study end date and replace their CSVs in the configured raw-data folder. SVXY starts on October 4, 2011, and SVIX on March 30, 2022. SVIX retains full price precision, adjusted Close, volume, and corporate-action columns for the checks in Notebook 12.

In [ ]:
# SVIX retains the columns and precision of the original study input.
SVIX_COLUMNS = OUTPUT_COLUMNS + [
    "Adj Close", "Volume", "Dividends", "Stock Splits", "Capital Gains",
]

def download_etf_data(symbol, start_date, output_file, *, retain_details=False):
    print(f"Downloading {symbol} historical data...")
    yahoo_end_date = END_DATE + pd.Timedelta(days=1)
    data = yf.Ticker(symbol).history(
        start=start_date.strftime("%Y-%m-%d"),
        end=yahoo_end_date.strftime("%Y-%m-%d"),
        interval="1d",
        auto_adjust=False,
        actions=retain_details,
    )
    if data.empty:
        raise ValueError(f"No historical data returned for {symbol}.")

    if data.index.tz is not None:
        data.index = data.index.tz_localize(None)
    data = data.reset_index().rename(columns={data.index.name or "index": "Date"})
    data["Date"] = pd.to_datetime(data["Date"], errors="coerce")
    data = data.loc[data["Date"].between(start_date, END_DATE)].dropna(subset=OUTPUT_COLUMNS)
    data = data.sort_values("Date").reset_index(drop=True)
    if data.empty:
        raise ValueError(f"No usable historical data returned for {symbol}.")

    if retain_details:
        # Yahoo Finance may omit this column for an ETF without capital gains.
        if "Capital Gains" not in data:
            data["Capital Gains"] = 0.0
        data = data[SVIX_COLUMNS].copy()
    else:
        data = data[OUTPUT_COLUMNS].copy()
        price_columns = ["Open", "High", "Low", "Close"]
        data[price_columns] = data[price_columns].round(6)

    data["Date"] = data["Date"].dt.strftime("%Y-%m-%d")
    data.to_csv(OUTPUT_FOLDER / output_file, index=False, encoding="utf-8")
    print(f"Saved {symbol} | Observations: {len(data):,}")
    return data

svxy_data = download_etf_data("SVXY", pd.Timestamp("2011-10-04"), "SVXY_data.csv")
svix_data = download_etf_data(
    "SVIX", pd.Timestamp("2022-03-30"), "SVIX_data.csv", retain_details=True,
)


### 5. Data Validation & Summary Statistics
Compile the downloaded datasets into a summary table to verify that all date ranges align with the Capstone requirements and that the expected number of observations were successfully captured.

In [ ]:
# Compile and display a summary dataframe of all downloaded data
datasets = {**historical_data, "SVXY": svxy_data, "SVIX": svix_data}

summary = pd.DataFrame([
    {
        "Dataset": name,
        "Observations": len(data),
        "First Date": data["Date"].iloc[0],
        "Last Date": data["Date"].iloc[-1]
    }
    for name, data in datasets.items()
])

print("\nData Retrieval Successfully Completed.")
display(summary)